# Xenium Human Lung Cancer (FFPE) workflow

## Input data: two lung cancer FFPE datasets from 10x Genomics (CC BY 4.0)

| | **Dataset A (start here)** | **Dataset B (scale-up)** |
|---|---|---|
| Platform | Xenium v1, Onboard Analysis 2.0.0, multimodal segmentation | Xenium Prime 5K, Onboard Analysis 3.0.0 |
| Gene panel | Human Multi-Tissue & Cancer (377 genes + 20 negative-control probes) | 5K Pan Tissue & Pathways (~5,100 genes) |
| Tissue area | 30.3 mm² | 71.1 mm² |
| Cells | 162,254 | 278,328 |
| High-quality transcripts (Q20) | 10.6 M | 149.4 M |
| Median transcripts per cell | 46 | 242 |
| Median genes per cell | 29 | 196 |
| Share of transcripts assigned to cells | 86.6% | 82.0% |
| Download size (zip) | ~8.0 GB | ~22.6 GB |

**Download base URLs**
- Dataset A: `https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_humanLung_Cancer_FFPE/` ([dataset page](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard))
- Dataset B: `https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/` ([dataset page](https://www.10xgenomics.com/datasets/xenium-human-lung-cancer-post-xenium-technote))

### Dataset A's other QC numbers
- 87.4% of transcripts pass the Q20 quality filter.
- The negative-control rate is very low (about 0.05 false-positive transcripts per cell).
- Cell segmentation used three methods: 33% of cells from a boundary stain, 65% from an interior stain, and 1.5% by expanding the nucleus outline.

### Its panel covers what a TIME analysis needs
- **T cells:** CD3D/E, CD8A, CD4, FOXP3, TRAC
- **Checkpoints:** PDCD1, CD274 (PD-L1), CTLA4, LAG3, HAVCR2
- **Myeloid:** CD68, CD163, MARCO, TREM2
- **B and plasma cells:** MS4A1, CD79A, MZB1
- **Tumour and epithelium:** EPCAM, KRT7, EGFR, ERBB2, MET
- **Stroma:** PDGFRA/B, ACTA2
- **Proliferation:** MKI67, TOP2A


### Proposed pipeline

1. Download Dataset A.
2. Load it with spatialdata-io and explore the raw files with pyarrow.
3. Run QC.
4. Normalise and cluster.
5. Annotate cell types.
6. Spatial analysis (neighbourhood enrichment, niches, immune infiltration).
7. Align the H&E image.
8. Optionally, repeat on Dataset B.

## Step 1: Download Dataset A

Raw data goes to `~/data/xenium_lung/`, **outside OneDrive**, so multi-GB files don't clog sync.
The cell is safe to re-run: finished files are skipped and interrupted downloads resume where they stopped.

- `DOWNLOAD_HE = True` also fetches the ~1.2 GB matched H&E image, used in Step 7 to map cells onto pathology regions (tumour, stroma, TLS). Set it to `False` only if disk space is tight; re-running later fetches just the missing image.
- Needs ~8 GB for the zip plus roughly the same again once unzipped; set `DELETE_ZIP_AFTER_UNZIP = True` to reclaim the zip space.


In [1]:
import shutil
import zipfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

# ---- Settings ---------------------------------------------------------------
DATA_ROOT = Path("~/data/xenium_lung").expanduser()   # outside OneDrive
DOWNLOAD_HE = True              # ~1.2 GB matched H&E image for Step 7 alignment
DELETE_ZIP_AFTER_UNZIP = False  # set True to free ~8 GB once extracted

SAMPLE = "Xenium_V1_humanLung_Cancer_FFPE"
BASE_URL = f"https://cf.10xgenomics.com/samples/xenium/2.0.0/{SAMPLE}"

FILES = [
    f"{SAMPLE}_metrics_summary.csv",
    f"{SAMPLE}_gene_panel.json",
    f"{SAMPLE}_he_imagealignment.csv",
    f"{SAMPLE}_outs.zip",
]
if DOWNLOAD_HE:
    FILES.append(f"{SAMPLE}_he_image.ome.tif")

DATA_DIR = DATA_ROOT / SAMPLE
# ask for raw bytes so byte counts match on-disk sizes (no gzip transfer encoding)
NO_GZIP = {"Accept-Encoding": "identity"}
DATA_DIR.mkdir(parents=True, exist_ok=True)


def remote_size(url):
    r = requests.head(url, headers=NO_GZIP, allow_redirects=True, timeout=60)
    r.raise_for_status()
    return int(r.headers.get("content-length", 0))


def download(url, dest):
    """Resumable download: skips complete files, continues partial ones."""
    total = remote_size(url)
    have = dest.stat().st_size if dest.exists() else 0
    if total and have == total:
        print(f"[skip] {dest.name} already complete ({total / 1e9:.2f} GB)")
        return
    if have > total:  # corrupt / mismatched partial, start over
        dest.unlink()
        have = 0

    headers = {**NO_GZIP, **({"Range": f"bytes={have}-"} if have else {})}
    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        if have and r.status_code != 206:  # server ignored Range, restart
            have = 0
        mode = "ab" if have else "wb"
        with open(dest, mode) as f, tqdm(
            total=total, initial=have, unit="B", unit_scale=True, desc=dest.name
        ) as bar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))

    if dest.stat().st_size != total:
        raise IOError(f"{dest.name}: size mismatch, re-run this cell to resume")


# ---- Disk space check -------------------------------------------------------
needed = sum(remote_size(f"{BASE_URL}/{f}") for f in FILES)
free = shutil.disk_usage(DATA_ROOT).free
print(f"Download size: {needed / 1e9:.1f} GB | free disk: {free / 1e9:.1f} GB")
if free < 2.2 * needed:  # zip + extracted copy + headroom
    print("WARNING: disk space is tight for download + unzip.")

# ---- Download ---------------------------------------------------------------
for fname in FILES:
    download(f"{BASE_URL}/{fname}", DATA_DIR / fname)

# ---- Unzip the output bundle -------------------------------------------------
zip_path = DATA_DIR / f"{SAMPLE}_outs.zip"
OUTS_DIR = DATA_DIR / "outs"
if not (OUTS_DIR / "experiment.xenium").exists():
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.infolist()
        # strip a single top-level folder (e.g. "outs/") if the zip has one
        tops = {m.filename.split("/")[0] for m in members}
        prefix = tops.pop() + "/" if len(tops) == 1 else ""
        OUTS_DIR.mkdir(exist_ok=True)
        for m in tqdm(members, desc="unzip"):
            rel = m.filename[len(prefix):]
            if not rel:
                continue
            target = OUTS_DIR / rel
            if m.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(m) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    if DELETE_ZIP_AFTER_UNZIP:
        zip_path.unlink()
else:
    print("[skip] outs/ already extracted")

# ---- Inventory (relative paths only) -----------------------------------------
print(f"\nContents of {SAMPLE}/outs:")
for item in sorted(OUTS_DIR.iterdir()):
    if item.is_file():
        size = item.stat().st_size
    else:
        size = sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
    print(f"  {item.name:<40} {size / 1e6:>10.1f} MB")


Download size: 9.2 GB | free disk: 47.4 GB


Xenium_V1_humanLung_Cancer_FFPE_metrics_summary.csv:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_gene_panel.json:   0%|          | 0.00/209k [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_he_imagealignment.csv:   0%|          | 0.00/126 [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_outs.zip:   0%|          | 0.00/7.99G [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_he_image.ome.tif:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

unzip:   0%|          | 0/26 [00:00<?, ?it/s]


Contents of Xenium_V1_humanLung_Cancer_FFPE/outs:
  analysis.tar.gz                                32.1 MB
  analysis.zarr.zip                               1.4 MB
  analysis_summary.html                          10.4 MB
  aux_outputs.tar.gz                             50.2 MB
  cell_boundaries.csv.gz                         38.4 MB
  cell_boundaries.parquet                        14.4 MB
  cell_feature_matrix.h5                          6.7 MB
  cell_feature_matrix.tar.gz                     23.0 MB
  cell_feature_matrix.zarr.zip                    9.1 MB
  cells.csv.gz                                    8.8 MB
  cells.parquet                                   2.9 MB
  cells.zarr.zip                                147.1 MB
  experiment.xenium                               0.0 MB
  gene_panel.json                                 0.2 MB
  metrics_summary.csv                             0.0 MB
  morphology.ome.tif                           4762.9 MB
  morphology_focus                   

In [ ]:
where the location of the data in my mac
